# GPU Environment Test

In [1]:
import torch
from torch import nn
import math

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda:0


# Reference BiKA Linear and Conv2d Implementation for Test

In [2]:
def ref_bika_linear(x, w, b):
    B, I = x.shape
    O, I2 = w.shape
    assert I == I2 and b.shape == w.shape
    # (B,O,I)
    xb = x[:, None, :] + b[None, :, :]
    z  = xb * w[None, :, :]
    # sign: z>=0 -> +1 else -1
    s  = torch.where(z >= 0, torch.ones_like(z), -torch.ones_like(z))
    return s.sum(dim=2)  # (B,O)

In [3]:
def ref_bika_conv2d(x, w, b, padding=(0,0), stride=(1,1)):
    B, C, H, W = x.shape
    O, C2, K, K2 = w.shape
    assert C == C2 and K == K2 and b.shape == w.shape
    ph, pw = padding if isinstance(padding, tuple) else (padding, padding)
    sh, sw = stride if isinstance(stride, tuple) else (stride, stride)
    Ho = (H + 2*ph - K)//sh + 1
    Wo = (W + 2*pw - K)//sw + 1

    # zero padding
    xp = torch.zeros(B, C, H + 2*ph, W + 2*pw, device=x.device, dtype=x.dtype)
    xp[:, :, ph:ph+H, pw:pw+W] = x

    y = torch.empty(B, O, Ho, Wo, device=x.device, dtype=x.dtype)
    for b0 in range(B):
        for o in range(O):
            for ho in range(Ho):
                for wo in range(Wo):
                    acc = 0.0
                    hbase = ho*sh
                    wbase = wo*sw
                    for c in range(C):
                        for kh in range(K):
                            for kw in range(K):
                                xv = xp[b0, c, hbase+kh, wbase+kw]
                                bb = b[o, c, kh, kw]
                                ww = w[o, c, kh, kw]
                                z  = (xv + bb) * ww
                                acc += 1.0 if z >= 0 else -1.0
                    y[b0, o, ho, wo] = acc
    return y

# Import BiKA Library

In [4]:
from bika import BiKA_Linear, BiKA_Conv2d

# BiKA_Linear Test

In [5]:
def test_linear_small_exact():
    torch.manual_seed(0)
    B, I, O = 2, 4, 3
    x = torch.tensor([
        [ 0.0,  1.0, -2.0,  3.0],
        [-1.0,  0.0,  2.0, -3.0],
    ], device=device, dtype=torch.float32)

    w = torch.tensor([
        [ 1.0, -1.0,  2.0, -2.0],
        [-1.5, 0.5,  -0.5,  1.5],
        [ 0.1, 0.2,   0.3,  0.4],
    ], device=device, dtype=torch.float32)
    b = torch.tensor([
        [ 0.0,  0.5, -0.5,  0.0],
        [ 0.2, -0.2,  0.2, -0.2],
        [-1.0,  1.0, -1.0,  1.0],
    ], device=device, dtype=torch.float32)

    y_ref = ref_bika_linear(x, w, b)

    m = BiKA_Linear(I, O, bias=True).to(device)
    with torch.no_grad():
        m.weight.copy_(w)
        m.bias.copy_(b)
    y_bika = m(x)

    diff = (y_ref - y_bika).abs().max().item()
    print("[Linear small] max abs diff:", diff)
    assert torch.equal(y_ref, y_bika), "  ❌ Linear forward mismatch!"
    print("  ✅ Linear small exact match")

In [6]:
def test_linear_random_trials(trials=3):
    torch.manual_seed(42)
    for t in range(trials):
        B, I, O = 3, 5, 4
        x = torch.randn(B, I, device=device, dtype=torch.float32)
        w = torch.randn(O, I, device=device, dtype=torch.float32)
        b = torch.randn(O, I, device=device, dtype=torch.float32)
        y_ref = ref_bika_linear(x, w, b)

        m = BiKA_Linear(I, O, bias=True).to(device)
        with torch.no_grad():
            m.weight.copy_(w)
            m.bias.copy_(b)
        y_bika = m(x)

        diff = (y_ref - y_bika).abs().max().item()
        print(f"[Linear rand {t}] max abs diff:", diff)
        assert torch.equal(y_ref, y_bika), f"Linear forward mismatch on trial {t}"
    print("  ✅ Linear random trials all matched")

# BiKA_Conv2d Test

In [7]:
def test_conv2d_basic():
    torch.manual_seed(0)
    B, C, H, W = 2, 2, 6, 5
    O, K = 3, 3
    padding = (1, 1)
    stride  = (1, 1)

    x = torch.randn(B, C, H, W, device=device, dtype=torch.float32)
    w = torch.randn(O, C, K, K, device=device, dtype=torch.float32)
    b = torch.randn(O, C, K, K, device=device, dtype=torch.float32)

    y_ref  = ref_bika_conv2d(x, w, b, padding=padding, stride=stride)

    m = BiKA_Conv2d(C, O, K, padding=padding, stride=stride, bias=True).to(device)
    with torch.no_grad():
        m.weight.copy_(w)
        m.bias.copy_(b)
    y_bika = m(x)

    print("[Conv2d basic] shapes:", y_ref.shape, y_bika.shape)
    diff = (y_ref - y_bika).abs().max().item()
    print("[Conv2d basic] max abs diff:", diff)
    assert torch.equal(y_ref, y_bika), "  ❌ Conv2d forward mismatch!"
    print("  ✅ Conv2d basic exact match")

In [8]:
def test_conv2d_stride_padding():
    torch.manual_seed(1)
    B, C, H, W = 1, 3, 7, 8
    O, K = 2, 3
    padding = (1, 2)
    stride  = (2, 3)

    x = torch.zeros(B, C, H, W, device=device, dtype=torch.float32)
    x[:, :, 2:5, 3:7] = torch.tensor(1.0, device=device)

    w = torch.tensor(0.5, device=device).expand(O, C, K, K).clone()
    b = torch.tensor(-0.3, device=device).expand(O, C, K, K).clone()
    w[0, :, 1, 1] = -0.7
    b[1, :, 0, 2] =  0.9

    y_ref  = ref_bika_conv2d(x, w, b, padding=padding, stride=stride)

    m = BiKA_Conv2d(C, O, K, padding=padding, stride=stride, bias=True).to(device)
    with torch.no_grad():
        m.weight.copy_(w)
        m.bias.copy_(b)
    y_bika = m(x)

    print("[Conv2d stride/pad] shapes:", y_ref.shape, y_bika.shape)
    diff = (y_ref - y_bika).abs().max().item()
    print("[Conv2d stride/pad] max abs diff:", diff)
    assert torch.equal(y_ref, y_bika), "  ❌ Conv2d forward mismatch (stride/padding case)!"
    print("  ✅ Conv2d stride>1 & asym padding exact match")

# Output Range Test

In [9]:
def test_sign_boundary_z_eq_0():
    x = torch.tensor([[0.0, 1.0]], device=device)
    w = torch.tensor([[1.0, 2.0]], device=device)      
    b = torch.tensor([[0.0, -1.0]], device=device)    
    y_ref  = ref_bika_linear(x, w, b)                  
    m = BiKA_Linear(2, 1, bias=True).to(device)
    with torch.no_grad():
        m.weight.copy_(w)
        m.bias.copy_(b)
    y_bika = m(x)
    print("[Boundary z=0] Linear:", y_ref.item(), y_bika.item())
    assert torch.equal(y_ref, y_bika), "  ❌ Sign boundary (z==0) mismatch!"
    print("  ✅ sign(z==0) treated as +1")

# Main Test

In [10]:
test_linear_small_exact()
test_linear_random_trials(trials=3)
test_conv2d_basic()
test_conv2d_stride_padding()
test_sign_boundary_z_eq_0()

[Linear small] max abs diff: 0.0
  ✅ Linear small exact match
[Linear rand 0] max abs diff: 0.0
[Linear rand 1] max abs diff: 0.0
[Linear rand 2] max abs diff: 0.0
  ✅ Linear random trials all matched
[Conv2d basic] shapes: torch.Size([2, 3, 6, 5]) torch.Size([2, 3, 6, 5])
[Conv2d basic] max abs diff: 0.0
  ✅ Conv2d basic exact match
[Conv2d stride/pad] shapes: torch.Size([1, 2, 4, 4]) torch.Size([1, 2, 4, 4])
[Conv2d stride/pad] max abs diff: 0.0
  ✅ Conv2d stride>1 & asym padding exact match
[Boundary z=0] Linear: 2.0 2.0
  ✅ sign(z==0) treated as +1
